# Classification - Logistic Regression

The previous lecture introduced classification concepts: why linear regression fails for categorical outcomes, the logistic function, and the idea of modeling probabilities rather than values. This notebook puts those ideas into practice with sklearn's `LogisticRegression`.

## Setup

Load the packages and configure environment.

In [ ]:
import numpy as np
import pandas as pd

Using the `Default` dataset, which predicts the likelihood of a person defaulting on credit card payments based on their balance, income, and student status (yes / no).

In [ ]:
# download the data set directly from the web using pandas
url = "https://raw.githubusercontent.com/olearydj/INSY7120/refs/heads/main/notebooks/data/Default.csv"
default = pd.read_csv(url)

default.head()

## Simple Logistic Regression

First, we'll fit a simple logistic regression with one predictor to see how classification differs from regression. We predict `default` based only on credit card `balance`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, confusion_matrix

Get `X` and `y`. Note that the method below for `y` is exactly equivalent to using `get_dummies` as we've done before:

```python
y = pd.get_dummies(default['default'], drop_first=True, dtype=int)
```

For this simple binary case, the code below creates a boolean array based on the comparison `default == 'Yes'`. In Python `True` is equivalent to `1` and `False` to zero, and `astype(int)` converts them to that representation.

In [ ]:
# Extract feature and target
X = default[['balance']].values  # We need 2D array for sklearn
y = (default['default'] == 'Yes').astype(int)  # Convert to binary 0/1

# use value_counts to see class counts
y.value_counts()

There are 10,000 observations, 9667 of which are False (96.7%), and 333 are True (3.3%).

For this example, we'll use simple train-test split validation.

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

We use a Pipeline to handle scaling, just as we did for regularization and KNN. Logistic regression needs standardization for the same reason Ridge and Lasso do - *sklearn's LogisticRegression applies L2 (Ridge) regularization by default*, and regularization requires features on the same scale to penalize them fairly.

Unlike linear regression, which fundamentally underfits most real relationships and only needs regularization (Ridge, Lasso) when we increase complexity (more predictors, polynomial / interaction terms), logistic regression is naturally prone to overfitting. When features are correlated or classes are well-separated, coefficients can grow without bound as the model chases perfect separation. L2 is the most common regularization in practice and stabilizes the optimization - so sklearn enables it by default.

In [ ]:
# Pipeline handles scaling + fitting in one object
model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_train, y_train)

Now we make predictions. Classification introduces two prediction methods: `predict` generates class labels (0 or 1) while `predict_proba` generates the underlying probabilities. sklearn's binary classifiers use a threshold of 0.5 for label assignment - observations with a predicted probability above 50% get the positive label.

In [ ]:
# Make predictions
y_pred = model.predict(X_test)
y_pred_prob = model.predict_proba(X_test)[:, 1]

Let's look at the relationship between probabilities, predictions, and the true labels.

In [ ]:
# Create a DataFrame to show samples with predictions and probabilities

# Get a sample of predictions
results_df = pd.DataFrame({
    'actual': y_test,
    'predicted': y_pred,
    'probability': y_pred_prob
})

# Sort by probability to show examples around the threshold
results_df = results_df.sort_values('probability')

# Select informative examples (some below and some above the threshold)
threshold_examples = pd.concat([
    # Examples just below threshold (predict 0)
    results_df[(results_df['probability'] > 0.3) & (results_df['probability'] < 0.5)].head(3),
    # Examples just above threshold (predict 1)
    results_df[(results_df['probability'] >= 0.5) & (results_df['probability'] < 0.7)].head(3)
])

# Display with formatting
pd.set_option('display.precision', 4)
print("Prediction Examples (threshold = 0.5):")
print(threshold_examples)

What if we don't want to use a 50% threshold?

***

##### Answer

sklearn's `predict` always uses 0.5, but that is not always the right threshold. In problems where one type of error is more costly than another (missing a fraud vs. a false alarm), we may want to lower or raise the threshold. We will cover threshold tuning in a later lecture on classification metrics.

***

### Interpreting the Predictions

To summarize and evaluate classification predictions, sklearn provides several functions, including `confusion_matrix` and `classification_report`:

In [ ]:
# Evaluate the model
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

These predictions are summarized above as a **Confusion Matrix**. In the case of binary classification it looks like this:

|            |                  |        **Predicted**         |        **Predicted**        |
| ---------- | ---------------- | :--------------------------: | :-------------------------: |
|            |                  |       **Negative (0)**       |      **Positive (1)**       |
| **Actual** | **Negative (0)** | True Negative (TN)<br>(2896) | False Positive (FP)<br>(10) |
| **Actual** | **Positive (1)** | False Negative (FN)<br>(70)  | True Positive (TP)<br>(24)  |

This table follows the scikit-learn format where:
- Rows represent actual values (0=No Default, 1=Default), the training labels
- Columns represent predicted values (0=No Default, 1=Default)

The four cells correspond to:

- **Top-Left (2896)**: True Negatives (TN) - Correctly predicted as "No Default"
- **Top-Right (10)**: False Positives (FP) - Incorrectly predicted as "Default"
- **Bottom-Left (70)**: False Negatives (FN) - Incorrectly predicted as "No Default"
- **Bottom-Right (24)**: True Positives (TP) - Correctly predicted as "Default"

This provides a complete picture of how the model's predictions align with the actual values, showing where the model succeeds and where it makes errors.

How would you interpret these results?

***

##### Answer

The data has many more observations with a negative label (default = no) than positive. The first row of the CM shows 2896 + 10 = 2906 actual negatives and only 70 + 24 = 94 actual positives. That's a total of 3,000 observations, which matches the 30% test split we used.

If we think only in terms of the number of correct predictions, which are on the diagonal, this does quite well: (2896 + 24) / 3000 = 0.9733 → 97.3% correct.

It is also very good at identifying non-defaults (first row): 2896 / (2896 + 10) = 0.9966 → 99.7% correct.

Struggles to identify actual defaults (second row): 24 / (70 + 24) = 0.2553 → 25.5% correct.

Two types of errors, each with different frequencies and implications:

- False Positives (10) - predicting default when there is none
- False Negatives (70) - missing actual defaults

In these results, False Negatives are 7x more common than False Positives. In a lending business, failing to identify potential defaults (FNs) is typically more costly than incorrectly flagging bad borrowers (FPs). The high number of FNs suggests the model might not adequately address the business need despite seeming very accurate. Poor discriminator for this scenario.

Why might this model be biased towards predicting no default?

***

### Visualize the Results

The actual and predicted dots should both appear on the 1.0 and 0.0 lines. They are shown stacked slightly above and below for clarity.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.figure(figsize=(12, 7))

# Create small vertical offsets for clarity
offset_actual = 0.02
offset_pred = -0.02

# Plot actual values (blue dots)
plt.scatter(X_test, y_test + offset_actual, color='blue', alpha=0.5, label='Actual')

# Plot predicted values (red dots)
plt.scatter(X_test, y_pred + offset_pred, color='red', alpha=0.5, label='Predicted')

# Plot the predicted probability curve
balance_range = np.linspace(X[:, 0].min(), X[:, 0].max(), 300).reshape(-1, 1)
y_prob = model.predict_proba(balance_range)[:, 1]
plt.plot(balance_range, y_prob, 'g-', label='P(Default)')

# Decision boundary: horizontal at P=0.5, vertical at the balance where P crosses 0.5
plt.axhline(y=0.5, color='k', linestyle='--', alpha=0.5, label='Decision Boundary (P=0.5)')
threshold_idx = np.argmin(np.abs(y_prob - 0.5))
threshold_balance = balance_range[threshold_idx, 0]
plt.axvline(x=threshold_balance, color='k', linestyle='--', alpha=0.5)

# Annotate the threshold balance value
plt.text(threshold_balance + 50, 0.55, f'${threshold_balance:,.0f}',
         fontsize=12, ha='left')

# Create annotation boxes for False Negatives and False Positives
fn_rect = patches.Rectangle((1000, 0.99), 930, 0.06, linewidth=2, edgecolor='r', facecolor='none')
plt.gca().add_patch(fn_rect)
plt.text(1500, 1.08, "False Negatives", color='red', fontsize=16, ha='center')

fp_rect = patches.Rectangle((1955, -0.01), 600, 0.06, linewidth=2, edgecolor='r', facecolor='none')
plt.gca().add_patch(fp_rect)
plt.text(2250, -0.07, "False Positives", color='red', fontsize=16, ha='center')

plt.ylim(-0.15, 1.15)
plt.xlabel('Credit Card Balance')
plt.ylabel('Default Probability')
plt.title('Logistic Regression: Predicting Default based on Balance')
plt.legend()
plt.tight_layout()
plt.show()

We can observe a few things here:

- The decision boundary (dashed line) at 0.5 probability corresponds to a balance of approximately \$2000, where the model transitions from predicting "No Default" to "Default".
- Most accounts with balances below \$2k have low default probabilities and are classified as non-defaults (red dots at bottom)
- Most accounts with balances above \$2k have high default probabilities and are classified as defaults (red dots at top)
- There are several misclassifications evident:
  - False negatives (actual = default, prediction = non-default): blue dots at y=1 without matching red dot
  - False positives (actual = non-default, prediction = default): blue dots at y=0 without matching red dot

Most FNs occur with balances between ≈\$1k and 2k. FPs are less common and occur at higher balances.

### Interpret Model Coefficients

Extract model coefficients from the fitted model and interpret. Since we used a Pipeline, we access the fitted components via `named_steps`.

In [ ]:
# Access the fitted model inside the Pipeline
logreg = model.named_steps['logisticregression']
scaler = model.named_steps['standardscaler']

print(f"Intercept: {logreg.intercept_[0]:.4f}")
print(f"Coefficient for balance: {logreg.coef_[0][0]:.4f}")

These don't match the results in ISL Chapter 4:

![](images/isl-tbl-4.1.png)

We standardized the features, so the coefficient is in standard-deviation units rather than dollars. In standardized scale:

> For every one *standard deviation* increase in `balance`, the *log-odds* of `default` increase by the coefficient value shown above.

This is useful for comparing the relative importance of multiple features, but not for answering "what happens when balance goes up by $1?" That is an *inference* question - interpreting what the coefficients mean in real-world terms. sklearn is a prediction library; it provides `predict` and `predict_proba` but has no built-in support for converting coefficients back to the original scale. We have to do that ourselves.

The idea is straightforward: substitute the standardization formula $X_{scaled} = (X - \mu) / \sigma$ into the model equation $\text{log-odds} = \beta_0 + \beta_1 X_{scaled}$ and collect terms. The result:

$$\beta_{1,orig} = \frac{\beta_1}{\sigma_X} \qquad \beta_{0,orig} = \beta_0 - \beta_1 \frac{\mu_X}{\sigma_X}$$

In [ ]:
# Convert coefficients back to original scale
coef_original = logreg.coef_[0][0] / scaler.scale_[0]
intercept_original = logreg.intercept_[0] - (logreg.coef_[0][0] * scaler.mean_[0] / scaler.scale_[0])

print(f"Coefficient in original scale: {coef_original:.4f}")
print(f"Intercept in original scale: {intercept_original:.4f}")
print(f"\nlog-odds(default) = {intercept_original:.4f} + {coef_original:.4f} x balance")

These results are close to ISL's values (-10.6513 + 0.0055 x balance). Small differences come from the train/test split and optimization method.

A caveat: these interpretations assume the logistic model is correct - that log-odds are truly linear in balance, that observations are independent, and that the model is well-specified. Coefficient interpretation is only as good as the model assumptions. Always keep the boundary between prediction and inference, and its implications to your claims, clear.

How does the derivation work? Expand for the full algebra.

***

##### Answer

A feature $X$ is standardized as $X_{scaled} = (X - \mu_X) / \sigma_X$. For a model fit on standardized data:

$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 X_{scaled}$$

Substituting:

$$\log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 \frac{X - \mu_X}{\sigma_X}$$

Rearranging:

$$\log\left(\frac{p}{1-p}\right) = \left[\beta_0 - \beta_1\frac{\mu_X}{\sigma_X}\right] + \frac{\beta_1}{\sigma_X}X$$

Reading off the terms: the slope in the original scale is $\beta_1 / \sigma_X$ and the intercept is $\beta_0 - \beta_1 \mu_X / \sigma_X$. The same principle applies per-feature in the multi-feature case.

***

## Multiple Logistic Regression

Now we use all three predictors: `balance`, `income`, and `student`. This introduces a new challenge - our data has mixed types. Numerical features need scaling; categorical features need encoding. So far our Pipelines have applied one transformer to all features. With mixed types, we need different transformations for different columns.

### When and What to Preprocess

In the Pipeline lecture we established *when* scaling is required: regularized models (Ridge, Lasso) and distance-based methods (KNN). `LogisticRegression` inherits the same requirement because sklearn applies L2 regularization by default. But scaling is only one kind of preprocessing. Here is a broader picture:

| Data Type | When | Transformer |
|-----------|------|-------------|
| Continuous numeric (balance, income) | Regularized or distance-based models | `StandardScaler` |
| Categorical, unordered (student, region) | Any model that expects numeric input | `OneHotEncoder` |
| Categorical, ordered (education level) | When order matters for the model | `OrdinalEncoder` |
| Missing values (any column with NaN) | Whenever data has gaps | `SimpleImputer` |

We have used `StandardScaler` since the regularization lecture. Now we add `OneHotEncoder` and `ColumnTransformer` to handle mixed-type data in a single Pipeline.

### `ColumnTransformer`

`ColumnTransformer` routes different columns to different transformers. It takes a list of tuples: `(name, transformer, columns)`.

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# define column types manually
categorical_cols = ['student']
numerical_cols = ['balance', 'income']

# Route numerical columns to StandardScaler, categorical to OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first'), categorical_cols)
    ])

`OneHotEncoder` does the same thing as `pd.get_dummies` - creates binary columns for each category - but it is Pipeline-compatible. `drop='first'` avoids the dummy variable trap, just as `drop_first=True` does in pandas.

The preprocessor plugs into a Pipeline as a step, just like `StandardScaler` did before. The Pipeline handles the rest: fit on training data, transform on test data, no leakage.

In [ ]:
# Build the full pipeline
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),  # column transformer for scaling / encoding
    ('classifier', LogisticRegression(random_state=42))
])

We need to redefine `X` - the simple model used only `balance` as a numpy array; now we need all three columns as a DataFrame so `ColumnTransformer` can route them by name.

In [ ]:
# Prepare X and y - now with all three features
X = default[numerical_cols + categorical_cols]
y = (default['default'] == 'Yes').astype(int)

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

In [ ]:
# Fit and predict
model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)
y_pred_prob = model_pipeline.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluate
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Adding income and student status changes very little - one prediction moves from FN to TP.

### Cross-Validation

A single train/test split is a starting point but we know from the regression unit that cross-validation gives a more reliable estimate. The Pipeline integrates with `cross_val_score` exactly as before.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model_pipeline, X, y, cv=cv, scoring='accuracy')

print(f"CV Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

### Regularization in Logistic Regression

sklearn's `LogisticRegression` applies L2 regularization by default. The penalty type is controlled by `l1_ratio`, a continuous value between 0 and 1: `l1_ratio=0` gives pure L2 (Ridge, the default), `l1_ratio=1` gives pure L1, and values in between include both penalty terms as a weighted sum - a combination known as Elastic Net:

$$\text{penalty} = \frac{1}{C} \left[ \texttt{l1\_ratio} \times \|w\|_1 + (1 - \texttt{l1\_ratio}) \times \frac{1}{2}\|w\|_2^2 \right]$$

`l1_ratio` controls the *mix* between L1 and L2; `C` controls the overall *strength*. So `l1_ratio=0.3` means 30% L1 + 70% L2, all scaled by `1/C`.

This is the same L1/L2 tradeoff we saw with Ridge and Lasso - it shrinks coefficients toward zero to prevent overfitting. `C` is the *inverse* of alpha: large `C` means *less* regularization, small `C` means more.

Why another name? Different communities developed the same idea independently. Ridge uses alpha ($\alpha$), the statistics literature uses lambda ($\lambda$), and logistic regression conventions use `C = 1/alpha`. They all control the same bias-variance tradeoff.

Since `LogisticRegression` is regularized by default, standardization is not optional - it is required for correctness. The Pipeline with `StandardScaler` is doing real work here, not just following convention.

We can tune `C` with `GridSearchCV`, using the same double-underscore naming pattern from the KNN lecture.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'classifier__C': [0.001, 0.01, 0.1, 1, 10, 100]}

grid_search = GridSearchCV(
    model_pipeline, param_grid, cv=cv, scoring='accuracy'
)
grid_search.fit(X, y)

print(f"Best C: {grid_search.best_params_['classifier__C']}")
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

### Interpreting Coefficients

Extract the coefficients from the fitted pipeline and display them.

In [ ]:
# Access fitted components
coefficients = model_pipeline.named_steps['classifier'].coef_[0]
intercept = model_pipeline.named_steps['classifier'].intercept_[0]

feature_names = numerical_cols + ['student_Yes']

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
})

print("Model Coefficients (standardized scale):")
print(coef_df)
print(f"\nIntercept: {intercept:.4f}")

The same caveats from the simple model apply here. These are standardized coefficients in log-odds space. Each coefficient represents the change in log-odds per standard deviation increase (for numerical features) or per category change (for encoded features), holding all other predictors constant.

- Balance has the strongest effect - higher balance substantially increases default log-odds
- Income has a smaller effect in the opposite direction
- Student status shows a *negative* association with default after controlling for balance and income

That last point is surprising - and leads us to confounding.

### Confounding Effects

The coefficient for student is negative in our multiple regression. This contradicts the single logistic regression result in ISL (pg 142, table 4.2), which shows a positive coefficient (0.4049). That positive result aligns with the raw default rates:

In [ ]:
student_balance = default.groupby('student')['balance'].mean()
print("Average balance by student status:")
print(student_balance)

default_by_student = default.groupby('student')['default'].apply(
    lambda x: (x == 'Yes').mean() * 100
)
print("\nDefault rate by student status:")
print(default_by_student.map(lambda x: f"{x:.2f}%"))

When we look at raw default rates without controlling for other factors, students appear to have higher default rates overall. This would lead to the incorrect conclusion that being a student directly increases default risk. However, our multiple regression shows the opposite - being a student is associated with a *lower* probability of default when we control for balance and income.

This apparent contradiction occurs because of *confounding*, where a variable affects both the predictor and the outcome, creating a misleading association between them. A confounder can make two variables appear related when they are not, or hide a real relationship.

1. Credit card balance is the confounding variable - it affects both student status (students tend to have higher balances) and default probability (higher balances increase default risk)
2. The negative coefficient reveals that after accounting for balance differences, students are actually more responsible borrowers than non-students in comparable financial situations
3. Without controlling for balance, the relationship between student status and default risk is obscured

We saw this same pattern in the regression unit with newspaper and radio advertising. Newspaper ad spend appeared to predict sales in SLR, but in MLR the effect disappeared - radio ads were the confounder, correlated with both newspaper spending and sales. Multiple regression makes this clear by controlling for the other variables.

### A Preview: The Class Imbalance Problem

Our model achieved 97% accuracy, but only caught 26% of actual defaults. This is not a bug - it is the central challenge of many real-world classification problems. Defaults, fraud, equipment failures, disease diagnoses - the events we most want to detect are often rare, sometimes occurring in less than 1% of observations. A model that simply predicts "no" for every observation would score 96.7% accuracy on this dataset.

When the positive class is rare, accuracy is misleading and the default 0.5 threshold is often too aggressive. The tools for addressing this - threshold tuning, precision-recall tradeoffs, and class-weighted models - are the focus of upcoming lectures on classification metrics.

## Summary

Key ideas from this notebook:

- `LogisticRegression` follows the same sklearn API (`fit`/`predict`) but outputs class probabilities via `predict_proba`
- `predict` applies a default threshold of 0.5 to convert probabilities to class labels
- `ColumnTransformer` routes different column types to different transformers inside a single Pipeline
- `OneHotEncoder` is the Pipeline-compatible version of `pd.get_dummies`
- sklearn's `LogisticRegression` applies L2 regularization by default (`l1_ratio=0`, `C=1.0`), which means standardization is required - same rule as Ridge and Lasso
- `l1_ratio` controls penalty type (0 = L2, 1 = L1); `C` controls strength as the inverse of alpha (large C = less regularization)
- The confusion matrix shows where the model succeeds and fails; precision, recall, and F1 will be unpacked in a later lecture
- Confounding can reverse apparent relationships - multiple regression controls for it by isolating each variable's independent effect